# Kickstarter Funding Prediction — NLP Embedding Baselines
### 4 Embeddings (Word2Vec, SBERT, DistilBERT, BERT) x 3 Models (XGBoost, CatBoost, Random Forest) = 12 Baseline Runs

This notebook merges each of the 4 NLP embedding sets (generated in `kickstarter_nlp_embeddings.ipynb`)
onto the existing tabular `ML_train.csv` / `ML_test.csv` features, and trains **baseline (default
hyperparameter)** XGBoost, CatBoost, and Random Forest models on each combination.

Conventions match the existing `kickstarter_all_models_ensembles_final.ipynb` baseline notebook:
- Target: `log_target`
- Dropped from X: `id`, `target_usd`, `log_target`, `goal_usd`
- Same `evaluate_model()` metrics function (MAE/MSE/RMSE/R2 on log scale and USD scale, plus RMSLE)
- Same baseline hyperparameters for XGBoost, CatBoost, Random Forest

**Output:** `kickstarter_nlp_embedding_baseline_12_results.csv`


## 0. Colab Setup — Mount Drive, Install Packages, Confirm GPU

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> **GPU (T4)**.

This cell mounts your Google Drive (so `ML_train.csv`, `ML_test.csv`, and the embedding
CSVs can be read from Drive), installs any packages missing from the default Colab image,
and confirms the T4 GPU is visible.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q catboost scikit-optimize

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 6.6 MB/s eta 0:00:00
/bin/bash: line 1: nvidia-smi: command not found


## 1. Imports

In [2]:
import os
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    r2_score
)

try:
    from xgboost import XGBRegressor
except ImportError:
    raise ImportError("xgboost is not installed. Run: pip install xgboost")

try:
    from catboost import CatBoostRegressor
except ImportError:
    raise ImportError("catboost is not installed. Run: pip install catboost")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42


In [ ]:
!ls -lah /content/drive/MyDrive

total 1.7G
drwx------ 2 root root 4.0K Aug 25 12:19 'Colab Notebooks'
drwx------ 4 root root 4.0K Aug 25 12:30  cse445
-rw------- 1 root root  179 Aug 28  2025  CV.gdoc
-rw------- 1 root root  179 May 11  2023 'faruq upwork.gdoc'
-rw------- 1 root root  98K Dec 22  2025  Ozodeck.inc.xlsx
-rw------- 1 root root  179 Dec 30  2022 'PSD to HTML gig.gdoc'
-rw------- 1 root root 5.1M Apr  6  2023  smart-it-certificate.png
-rw------- 1 root root 230M Mar 28  2025  verpack_backup_28march.zip
-rw------- 1 root root 1.4G Jul  6 14:37 'verwertung final backup.zip'
-rw------- 1 root root  179 Jan  5  2023 'wordpress elementor gig.gdoc'
-rw------- 1 root root 219K Jan 20  2025 'Zellyo payments.xlsx'


## 2. Load Tabular Train/Test and Embedding Files

In [3]:
# Update DRIVE_ROOT to wherever you uploaded the project folder in your Google Drive.
# Example: if your files are at
#   MyDrive/kickstarter_project/Dataset/ML dataset/ML_train.csv
# then DRIVE_ROOT should be "/content/drive/MyDrive/kickstarter_project"

DRIVE_ROOT = "/content/drive/MyDrive/cse445"

TRAIN_FILE = f"{DRIVE_ROOT}/ML dataset/ML_train.csv"
TEST_FILE = f"{DRIVE_ROOT}/ML dataset/ML_test.csv"

train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (16000, 81)
Test shape: (4000, 81)


In [4]:
EMBEDDING_PATH = f"{DRIVE_ROOT}/NLP dataset"

EMBEDDING_FILES = {
    "Word2Vec":   f"{EMBEDDING_PATH}/word2vec_embeddings.csv",
    "SBERT":      f"{EMBEDDING_PATH}/sbert_embeddings.csv",
    "DistilBERT": f"{EMBEDDING_PATH}/distilbert_embeddings.csv",
    "BERT":       f"{EMBEDDING_PATH}/bert_embeddings.csv",
    "TFIDF":      f"{EMBEDDING_PATH}/tfidf_embeddings.csv",
}

embeddings_raw = {}
for name, path in EMBEDDING_FILES.items():
    emb_df = pd.read_csv(path)
    embeddings_raw[name] = emb_df
    print(f"{name}: {emb_df.shape}")


Word2Vec: (20000, 101)
SBERT: (20000, 385)
DistilBERT: (20000, 769)
BERT: (20000, 769)
TFIDF: (20000, 5001)


## 3. PCA Reduction (fit on train rows only)

Each embedding set is reduced to 20 components to avoid dominating the ~80 existing tabular
features. PCA is fit only on the rows that belong to `train_df["id"]`, then applied to both
train and test rows to prevent leakage.


In [5]:
N_COMPONENTS = 20

train_ids = set(train_df["id"])
test_ids = set(test_df["id"])

embeddings_reduced = {}

for name, emb_df in embeddings_raw.items():
    feature_cols = [c for c in emb_df.columns if c != "id"]

    train_mask = emb_df["id"].isin(train_ids)
    test_mask = emb_df["id"].isin(test_ids)

    pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
    pca.fit(emb_df.loc[train_mask, feature_cols])

    reduced_all = pca.transform(emb_df[feature_cols])

    reduced_cols = [f"{name.lower()}_pca_{i}" for i in range(N_COMPONENTS)]
    reduced_df = pd.DataFrame(reduced_all, columns=reduced_cols)
    reduced_df.insert(0, "id", emb_df["id"].values)

    embeddings_reduced[name] = reduced_df

    explained = pca.explained_variance_ratio_.sum()
    print(f"{name}: reduced to {N_COMPONENTS} dims, explained variance = {explained:.3f}")


Word2Vec: reduced to 20 dims, explained variance = 0.883
SBERT: reduced to 20 dims, explained variance = 0.684
DistilBERT: reduced to 20 dims, explained variance = 0.768
BERT: reduced to 20 dims, explained variance = 0.727
TFIDF: reduced to 20 dims, explained variance = 0.162


## 4. Evaluation Helper (same as baseline notebook)

In [6]:
def evaluate_model(model_name, y_true_log, pred_log, actual_usd):
    pred_usd = np.expm1(pred_log)
    pred_usd = np.clip(pred_usd, a_min=0, a_max=None)

    mae_log = mean_absolute_error(y_true_log, pred_log)
    mse_log = mean_squared_error(y_true_log, pred_log)
    rmse_log = np.sqrt(mse_log)
    r2_log = r2_score(y_true_log, pred_log)

    mae_usd = mean_absolute_error(actual_usd, pred_usd)
    mse_usd = mean_squared_error(actual_usd, pred_usd)
    rmse_usd = np.sqrt(mse_usd)
    r2_usd = r2_score(actual_usd, pred_usd)

    rmsle = np.sqrt(mean_squared_log_error(actual_usd, pred_usd))

    metrics = {
        "Model":      model_name,
        "MAE_log":    mae_log,
        "MSE_log":    mse_log,
        "RMSE_log":   rmse_log,
        "R2_log":     r2_log,
        "MAE_USD":    mae_usd,
        "MSE_USD":    mse_usd,
        "RMSE_USD":   rmse_usd,
        "R2_USD":     r2_usd,
        "RMSLE":      rmsle
    }

    return metrics, pred_usd


## 5. Merge Function + Feature Setup

For each embedding, merge its reduced PCA columns onto `train_df` / `test_df` via `id`,
then build `X_train` / `X_test` / `y_train` / `y_test` using the same `DROP_FROM_X` convention
as the baseline notebook.


In [7]:
TARGET_RAW = "target_usd"
TARGET_LOG = "log_target"

DROP_FROM_X = [
    "id",
    "target_usd",
    "log_target",
    "goal_usd"
]

def build_features_for_embedding(embedding_name):
    reduced_df = embeddings_reduced[embedding_name]

    merged_train = train_df.merge(reduced_df, on="id", how="left")
    merged_test = test_df.merge(reduced_df, on="id", how="left")

    assert merged_train.shape[0] == train_df.shape[0], "Row count mismatch after merge (train)"
    assert merged_test.shape[0] == test_df.shape[0], "Row count mismatch after merge (test)"

    pca_cols = [c for c in reduced_df.columns if c != "id"]
    assert merged_train[pca_cols].isna().sum().sum() == 0, f"NaNs found after merging {embedding_name} (train)"
    assert merged_test[pca_cols].isna().sum().sum() == 0, f"NaNs found after merging {embedding_name} (test)"

    X_train = merged_train.drop(columns=DROP_FROM_X, errors="ignore")
    X_test = merged_test.drop(columns=DROP_FROM_X, errors="ignore")

    y_train = merged_train[TARGET_LOG].copy()
    y_test = merged_test[TARGET_LOG].copy()

    actual_usd = merged_test[TARGET_RAW].to_numpy()

    assert list(X_train.columns) == list(X_test.columns)

    return X_train, X_test, y_train, y_test, actual_usd


## 6. Baseline Model Definitions

Same default hyperparameters as the original baseline notebook (`kickstarter_all_models_ensembles_final.ipynb`).


In [8]:
USE_GPU = True  # set False to force CPU (e.g. if no GPU runtime is attached)

def make_random_forest():
    # scikit-learn's RandomForestRegressor has no GPU support; always runs on CPU
    return RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )

def make_xgboost():
    return XGBRegressor(
        n_estimators=800,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=1.0,
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        device="cuda" if USE_GPU else "cpu",
        random_state=RANDOM_STATE
    )

def make_catboost():
    return CatBoostRegressor(
        iterations=800,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=3.0,
        loss_function="RMSE",
        eval_metric="RMSE",
        task_type="GPU" if USE_GPU else "CPU",
        devices="0" if USE_GPU else None,
        random_seed=RANDOM_STATE,
        verbose=False
    )

MODEL_FACTORIES = {
    "Random Forest": make_random_forest,
    "XGBoost":       make_xgboost,
    "CatBoost":      make_catboost,
}


## 7. Run All 12 Combinations (5 Embeddings x 3 Models)

For each embedding, build features once, then train/evaluate all 3 models.



In [ ]:
all_results = []

for embedding_name in embeddings_reduced.keys():
    print(f"\n{'='*60}")
    print(f"Embedding: {embedding_name}")
    print(f"{'='*60}")

    X_train, X_test, y_train, y_test, actual_usd = build_features_for_embedding(embedding_name)
    print(f"Feature count: {X_train.shape[1]}")

    for model_name, factory in MODEL_FACTORIES.items():
        model = factory()

        start_time = time.time()

        if model_name == "XGBoost":
            model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        elif model_name == "CatBoost":
            model.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=False)
        else:
            model.fit(X_train, y_train)

        training_time = time.time() - start_time

        pred_log = model.predict(X_test)

        run_label = f"{embedding_name} + {model_name}"
        metrics, pred_usd = evaluate_model(run_label, y_test, pred_log, actual_usd)
        metrics["Embedding"] = embedding_name
        metrics["Base_Model"] = model_name
        metrics["Training_Time_Seconds"] = training_time

        all_results.append(metrics)

        print(f"  {model_name}: RMSLE={metrics['RMSLE']:.4f}, R2_log={metrics['R2_log']:.4f}, "
              f"time={training_time:.1f}s")

print(f"\nTotal runs completed: {len(all_results)}")



Embedding: Word2Vec
Feature count: 97
  Random Forest: RMSLE=2.1076, R2_log=0.5272, time=31.7s
  XGBoost: RMSLE=1.8436, R2_log=0.6373, time=3.5s
  CatBoost: RMSLE=1.9312, R2_log=0.6018, time=14.9s

Embedding: SBERT
Feature count: 97
  Random Forest: RMSLE=2.3169, R2_log=0.4286, time=29.3s
  XGBoost: RMSLE=2.2070, R2_log=0.4815, time=3.5s
  CatBoost: RMSLE=2.2491, R2_log=0.4616, time=17.9s

Embedding: DistilBERT
Feature count: 97
  Random Forest: RMSLE=2.1130, R2_log=0.5248, time=26.9s
  XGBoost: RMSLE=1.7487, R2_log=0.6745, time=4.1s
  CatBoost: RMSLE=1.8981, R2_log=0.6165, time=15.3s

Embedding: BERT
Feature count: 97
  Random Forest: RMSLE=1.9529, R2_log=0.5940, time=27.8s
  XGBoost: RMSLE=1.5290, R2_log=0.7508, time=3.6s
  CatBoost: RMSLE=1.6510, R2_log=0.7094, time=15.0s

Embedding: TFIDF
Feature count: 97
  Random Forest: RMSLE=2.3024, R2_log=0.4357, time=28.8s
  XGBoost: RMSLE=2.1318, R2_log=0.5161, time=3.4s
  CatBoost: RMSLE=2.1895, R2_log=0.4896, time=14.9s

Total runs comple

## 8. Combine and Display Results

In [ ]:
results_df = pd.DataFrame(all_results)

results_df = results_df[
    [
        "Embedding",
        "Base_Model",
        "Model",
        "MAE_log",
        "MSE_log",
        "RMSE_log",
        "R2_log",
        "MAE_USD",
        "MSE_USD",
        "RMSE_USD",
        "R2_USD",
        "RMSLE",
        "Training_Time_Seconds"
    ]
]

results_df = results_df.sort_values("RMSLE").reset_index(drop=True)
display(results_df)


,Embedding,Base_Model,Model,MAE_log,MSE_log,RMSE_log,R2_log,MAE_USD,MSE_USD,RMSE_USD,R2_USD,RMSLE,Training_Time_Seconds
0,BERT,XGBoost,BERT + XGBoost,1.127748,2.340780,1.529961,0.750841,14639.855619,1.022600e+10,101123.679816,0.133500,1.529040,3.611091
1,BERT,CatBoost,BERT + CatBoost,1.241084,2.729653,1.652166,0.709449,15430.808946,1.096431e+10,104710.598003,0.070939,1.650970,14.967670
2,DistilBERT,XGBoost,DistilBERT + XGBoost,1.288457,3.058206,1.748773,0.674477,15056.298390,1.042965e+10,102125.641500,0.116244,1.748727,4.055069
3,Word2Vec,XGBoost,Word2Vec + XGBoost,1.354563,3.407485,1.845937,0.637299,15717.656569,1.095446e+10,104663.563402,0.071773,1.843574,3.495053
4,DistilBERT,CatBoost,DistilBERT + CatBoost,1.425831,3.602832,1.898113,0.616506,15921.376032,1.108559e+10,105288.115512,0.060662,1.898058,15.265138
5,Word2Vec,CatBoost,Word2Vec + CatBoost,1.443599,3.740974,1.934160,0.601801,16302.894407,1.133752e+10,106477.770207,0.039315,1.931161,14.943990
6,BERT,Random Forest,BERT + Random Forest,1.480686,3.813992,1.952944,0.594029,16300.268785,1.139903e+10,106766.246928,0.034103,1.952944,27.847407
7,Word2Vec,Random Forest,Word2Vec + Random Forest,1.600187,4.441945,2.107592,0.527188,16646.583646,1.162816e+10,107833.940889,0.014688,2.107592,31.741519
8,DistilBERT,Random Forest,DistilBERT + Random Forest,1.590226,4.464834,2.113015,0.524752,16473.843356,1.149365e+10,107208.448672,0.026085,2.113015,26.907058
9,TFIDF,XGBoost,TFIDF + XGBoost,1.566486,4.545847,2.132099,0.516129,15895.421343,1.103861e+10,105064.772964,0.064643,2.131795,3.352528


In [ ]:
best_row = results_df.iloc[0]
print("Best embedding + model combination by RMSLE:")
print(best_row["Model"])
print(f"RMSLE:   {best_row['RMSLE']:.4f}")
print(f"R2_log:  {best_row['R2_log']:.4f}")
print(f"MAE_USD: \${best_row['MAE_USD']:,.2f}")


Best embedding + model combination by RMSLE:
BERT + XGBoost
RMSLE:   1.5290
R2_log:  0.7508
MAE_USD: \$14,639.86


## 9. Save Results

In [ ]:
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs"
import os as _os
_os.makedirs(OUTPUT_DIR, exist_ok=True)

results_df.to_csv(f"{OUTPUT_DIR}/kickstarter_nlp_embedding_baseline_12_results.csv", index=False)
print(f"Saved: {OUTPUT_DIR}/kickstarter_nlp_embedding_baseline_12_results.csv")


Saved: /content/drive/MyDrive/cse445/outputs/kickstarter_nlp_embedding_baseline_12_results.csv


## Summary

This notebook produced **12 baseline runs** (4 embeddings x 3 models, all default hyperparameters):

| Embedding | Models |
|---|---|
| Word2Vec | Random Forest, XGBoost, CatBoost |
| SBERT | Random Forest, XGBoost, CatBoost |
| DistilBERT | Random Forest, XGBoost, CatBoost |
| BERT | Random Forest, XGBoost, CatBoost |

**Next step (not part of this notebook):** apply hyperparameter tuning (RandomizedSearchCV,
GridSearchCV, BayesSearchCV) to each of these 12 combinations, and compare tuned vs. baseline
results, alongside the original tabular-only baseline from `kickstarter_all_models_ensembles_final.ipynb`.


## 6. Hyperparameter Tuning — 3 Search Methods (RandomizedSearchCV, HalvingGridSearchCV, BayesSearchCV)

Defines search spaces and a unified `make_search_estimator()` helper for all three tuning
methods, for each of the 3 models (RF, XGBoost, CatBoost):

- **RandomizedSearchCV** — samples `N_ITER_RANDOM` random combos from a wide param distribution.
  Fast, scales well to large search spaces.
- **HalvingGridSearchCV** — covers the *full* grid exhaustively in terms of parameter combos
  visited, but starts every candidate on a small subset of training data/resources, discards the
  worst-performing half (or `1/factor`) each round, and only lets survivors train on progressively
  more data. This is scikit-learn's recommended way to run grid search efficiently — full grid
  coverage without the wall-clock cost of plain `GridSearchCV`.
- **BayesSearchCV** — from `scikit-optimize`, uses Bayesian optimization to intelligently pick the
  next candidate based on prior results. Requires `pip install scikit-optimize`.


In [15]:
from sklearn.experimental import enable_halving_search_cv  # noqa: F401 — required to unlock HalvingGridSearchCV
from sklearn.model_selection import RandomizedSearchCV, HalvingGridSearchCV, KFold
from sklearn.metrics import make_scorer

try:
    from skopt import BayesSearchCV
    from skopt.space import Real, Integer, Categorical
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False
    print("scikit-optimize not installed — BayesSearchCV will be skipped. "
          "Run: pip install scikit-optimize")

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

# 3-fold instead of 5-fold cuts every fit count by 40% — Colab's free-tier CPU
# (often just 2 vCPUs) makes 5-fold CV prohibitively slow for CPU-only RF.
CV = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
N_ITER_RANDOM = 8  # candidates for RandomizedSearchCV (was 15)
N_ITER_BAYES = 6    # candidates for BayesSearchCV (was 12)
HALVING_FACTOR = 3  # each round keeps 1/factor of candidates, factor x more resources


In [16]:
# --- Wide distributions used by RandomizedSearchCV ---

xgb_param_dist = {
    "n_estimators":     [400, 600, 800, 1000, 1200],
    "learning_rate":    [0.01, 0.02, 0.03, 0.05, 0.08],
    "max_depth":        [4, 5, 6, 7, 8, 9],
    "min_child_weight": [1, 2, 3, 5, 7],
    "subsample":        [0.7, 0.8, 0.85, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.85, 0.9, 1.0],
    "reg_alpha":        [0, 0.01, 0.05, 0.1, 0.5],
    "reg_lambda":       [0.5, 1.0, 1.5, 2.0, 3.0],
}

catboost_param_dist = {
    "iterations":    [400, 600, 800, 1000, 1200],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "depth":         [4, 5, 6, 7, 8, 9],
    "l2_leaf_reg":   [1, 2, 3, 5, 7, 9],
}

rf_param_dist = {
    # trimmed down (was up to n_estimators=900, max_depth=None/unbounded) —
    # those combos are what caused ~108s/fit on Colab's limited CPU cores
    "n_estimators":      [150, 250, 350, 450],
    "max_depth":         [10, 15, 20, 25],
    "min_samples_split": [2, 4, 6, 8],
    "min_samples_leaf":  [1, 2, 3, 4],
    "max_features":      ["sqrt", "log2", 0.5, 0.7],
}


In [17]:
# --- Grid used by HalvingGridSearchCV ---
# HalvingGridSearchCV can afford the FULL grid (unlike plain GridSearchCV) because it
# starts every candidate on a small data subset and only promotes survivors to more data
# across successive rounds — so we reuse the wide param_dist grids from RandomizedSearchCV
# above instead of maintaining a separate hand-shrunk grid.

xgb_param_grid = xgb_param_dist
catboost_param_grid = catboost_param_dist
rf_param_grid = rf_param_dist


In [18]:
# --- Continuous/discrete search spaces used by BayesSearchCV ---

if SKOPT_AVAILABLE:
    xgb_bayes_space = {
        "n_estimators":     Integer(300, 1200),
        "learning_rate":    Real(0.01, 0.1, prior="log-uniform"),
        "max_depth":        Integer(4, 9),
        "min_child_weight": Integer(1, 7),
        "subsample":        Real(0.7, 1.0),
        "colsample_bytree": Real(0.7, 1.0),
        "reg_alpha":        Real(1e-3, 0.5, prior="log-uniform"),
        "reg_lambda":       Real(0.5, 3.0),
    }

    catboost_bayes_space = {
        "iterations":    Integer(300, 1200),
        "learning_rate": Real(0.01, 0.1, prior="log-uniform"),
        "depth":         Integer(4, 9),
        "l2_leaf_reg":   Real(1.0, 9.0),
    }

    rf_bayes_space = {
        "n_estimators":      Integer(300, 900),
        "max_depth":         Integer(5, 30),
        "min_samples_split": Integer(2, 8),
        "min_samples_leaf":  Integer(1, 4),
        "max_features":      Categorical(["sqrt", "log2"]),
    }


In [19]:
def make_base_estimator(model_name):
    if model_name == "XGBoost":
        return XGBRegressor(
            objective="reg:squarederror", eval_metric="rmse",
            tree_method="hist", device="cuda" if USE_GPU else "cpu",
            random_state=RANDOM_STATE,
        )
    elif model_name == "CatBoost":
        return CatBoostRegressor(
            loss_function="RMSE", eval_metric="RMSE",
            task_type="GPU" if USE_GPU else "CPU", devices="0" if USE_GPU else None,
            random_seed=RANDOM_STATE, verbose=False,
        )
    elif model_name == "Random Forest":
        # no GPU support in sklearn's RF; runs on CPU regardless of USE_GPU
        return RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE)
    else:
        raise ValueError(model_name)


def make_search_estimator(model_name, method):
    """method: 'random', 'grid' (HalvingGridSearchCV), or 'bayes'"""
    base = make_base_estimator(model_name)

    # GPU-backed XGBoost/CatBoost fits are already accelerated internally, so keep
    # search-level n_jobs=1 for them to avoid spawning multiple GPU-contending processes.
    # RF has no GPU acceleration, so let sklearn parallelize across CPU cores instead.
    # IMPORTANT: never set both search-level and estimator-level n_jobs=-1 —
    # nested parallelism causes thread contention and is SLOWER than sequential,
    # especially on Colab where CPU cores are limited (often just 2) even with a GPU attached.
    # RF has no GPU support, so it needs the CPU parallelism at the estimator level instead.
    if model_name == "Random Forest":
        search_n_jobs = 1
    else:
        search_n_jobs = 1

    if method == "random":
        param_dist = {"XGBoost": xgb_param_dist, "CatBoost": catboost_param_dist,
                       "Random Forest": rf_param_dist}[model_name]
        return RandomizedSearchCV(
            estimator=base, param_distributions=param_dist, n_iter=N_ITER_RANDOM,
            scoring=rmse_scorer, cv=CV, n_jobs=search_n_jobs,
            random_state=RANDOM_STATE, verbose=0, refit=True,
        )

    elif method == "grid":
        param_grid = {"XGBoost": xgb_param_grid, "CatBoost": catboost_param_grid,
                       "Random Forest": rf_param_grid}[model_name]
        return HalvingGridSearchCV(
            estimator=base, param_grid=param_grid,
            scoring=rmse_scorer, cv=CV, n_jobs=search_n_jobs,
            factor=HALVING_FACTOR, resource="n_samples", min_resources="exhaust",
            random_state=RANDOM_STATE, verbose=0, refit=True,
        )

    elif method == "bayes":
        if not SKOPT_AVAILABLE:
            return None
        search_space = {"XGBoost": xgb_bayes_space, "CatBoost": catboost_bayes_space,
                         "Random Forest": rf_bayes_space}[model_name]
        return BayesSearchCV(
            estimator=base, search_spaces=search_space, n_iter=N_ITER_BAYES,
            scoring=rmse_scorer, cv=CV, n_jobs=search_n_jobs,
            random_state=RANDOM_STATE, verbose=0, refit=True,
        )

    else:
        raise ValueError(method)


TUNING_METHODS = ["random", "grid", "bayes"] if SKOPT_AVAILABLE else ["random", "grid"]
TUNING_METHOD_LABELS = {
    "random": "RandomizedSearchCV",
    "grid":   "HalvingGridSearchCV",
    "bayes":  "BayesSearchCV",
}
print("Tuning methods to run:", [TUNING_METHOD_LABELS[m] for m in TUNING_METHODS])
print("USE_GPU:", USE_GPU)


Tuning methods to run: ['RandomizedSearchCV', 'HalvingGridSearchCV', 'BayesSearchCV']
USE_GPU: True


## 7. Run All 3 Tuning Methods for All 12 Combinations

3 tuning methods (RandomizedSearchCV, HalvingGridSearchCV, BayesSearchCV) x 4 embeddings x 3
models = up to 36 tuned runs (fewer if `scikit-optimize` isn't installed, since BayesSearchCV
is then skipped). For each embedding, features are built once and reused across all tuning
methods and models. `HalvingGridSearchCV` covers the full grid but at a fraction of the cost
of plain `GridSearchCV`, since it discards weak candidates early on small data subsets.


In [ ]:
tuned_results = []
best_params_log = {}

for embedding_name in embeddings_reduced.keys():
    print(f"\n{'='*60}")
    print(f"Embedding: {embedding_name}")
    print(f"{'='*60}")

    X_train, X_test, y_train, y_test, actual_usd = build_features_for_embedding(embedding_name)

    for model_name in MODEL_FACTORIES.keys():
        for method in TUNING_METHODS:
            method_label = TUNING_METHOD_LABELS[method]
            print(f"  Tuning {model_name} via {method_label}...")
            start_time = time.time()

            search = make_search_estimator(model_name, method)
            if search is None:
                print(f"    Skipped ({method_label} unavailable)")
                continue

            search.fit(X_train, y_train)

            best_model = search.best_estimator_
            tuning_time = time.time() - start_time

            pred_log = best_model.predict(X_test)

            run_label = f"{embedding_name} + {model_name} ({method_label})"
            metrics, pred_usd = evaluate_model(run_label, y_test, pred_log, actual_usd)
            metrics["Embedding"] = embedding_name
            metrics["Base_Model"] = model_name
            metrics["Tuning_Method"] = method_label
            metrics["Tuning_Time_Seconds"] = tuning_time
            metrics["Best_CV_RMSE_log"] = -search.best_score_

            tuned_results.append(metrics)
            best_params_log[f"{embedding_name}_{model_name}_{method}"] = dict(search.best_params_)

            print(f"    Best CV RMSE_log={-search.best_score_:.4f} | "
                  f"Test RMSLE={metrics['RMSLE']:.4f}, R2_log={metrics['R2_log']:.4f}, "
                  f"time={tuning_time:.1f}s")

print(f"\nTotal tuned runs completed: {len(tuned_results)}")



Embedding: Word2Vec
  Tuning Random Forest via RandomizedSearchCV...
    Best CV RMSE_log=2.0691 | Test RMSLE=2.0608, R2_log=0.5480, time=509.3s
  Tuning Random Forest via HalvingGridSearchCV...
    Best CV RMSE_log=2.0616 | Test RMSLE=2.0423, R2_log=0.5560, time=4803.7s
  Tuning Random Forest via BayesSearchCV...
    Best CV RMSE_log=2.1352 | Test RMSLE=2.1199, R2_log=0.5217, time=502.6s
  Tuning XGBoost via RandomizedSearchCV...
    Best CV RMSE_log=1.8666 | Test RMSLE=1.8338, R2_log=0.6415, time=464.1s
  Tuning XGBoost via HalvingGridSearchCV...


In [ ]:
tuned_df = pd.DataFrame(tuned_results)

tuned_df = tuned_df[
    [
        "Embedding", "Base_Model", "Tuning_Method", "Model",
        "MAE_log", "MSE_log", "RMSE_log", "R2_log",
        "MAE_USD", "MSE_USD", "RMSE_USD", "R2_USD",
        "RMSLE", "Best_CV_RMSE_log", "Tuning_Time_Seconds"
    ]
]
tuned_df = tuned_df.sort_values("RMSLE").reset_index(drop=True)
display(tuned_df)

best_by_method = tuned_df.sort_values("RMSLE").groupby("Tuning_Method").first()
print("\nBest result per tuning method:")
display(best_by_method[["Embedding", "Base_Model", "RMSLE", "R2_log"]])


tuned_df.to_csv(f"{OUTPUT_DIR}/kickstarter_nlp_embedding_tuned_all_methods_results.csv", index=False)

import json
with open(f"{OUTPUT_DIR}/kickstarter_nlp_embedding_best_params.json", "w") as f:
    json.dump(best_params_log, f, indent=2, default=str)

print(f"\nSaved: {OUTPUT_DIR}/kickstarter_nlp_embedding_tuned_all_methods_results.csv")
print(f"Saved: {OUTPUT_DIR}/kickstarter_nlp_embedding_best_params.json")
